# VLM text extraction vs OCR

Traditional OCR loses decision-relevant content on real social-media ad
screenshots. Measured on the RYZE ad, RapidOCR read `ISWEEKONLY` (a profile
avatar overlapped the "TH"), so the Urgency trigger never matched and the system
reported the ad as clean — the exact failure that motivated this.

This notebook tests whether a vision-language model recovers what OCR loses, and
produces the comparison table requested for the writeup.

**Scope of the VLM's role.** It transcribes and it names what the ad claims. It
never judges the ad, never names a tactic, and never decides whether a claim is
true. Classification stays with the fine-tuned DistilBERT; verification stays
with retrieval. The prompts below enforce that.

## Setup

In [1]:
!pip install openai rapidocr-onnxruntime paddleocr paddlepaddle pillow -q
print("installed")

installed


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [ ]:
import base64, io, json, re, time, os
from pathlib import Path

# ---- EDIT ----
OPENAI_API_KEY = ""          # or set the OPENAI_API_KEY env var
MODEL          = "gpt-4o"
BASE = Path(r"C:\Users\shash\OneDrive\Desktop\MSML641-Final Project-outputs")         # gpt-4o-mini is cheaper and usually enough
IMAGES = [
    BASE / "ad.png",
    BASE / "ad_1.png",
    BASE / "ad_12.png",
    BASE / "RYZE_AD.jpeg",
    BASE / "WhatsApp Image 2026-08-17 at 11.31.08 PM.jpeg",
    BASE / "WhatsApp Image 2026-08-17 at 11.31.09 PM.jpeg",
]
# --------------

IMAGES = [p for p in IMAGES if p.exists()]
print(f"{len(IMAGES)} image(s) found")
for p in IMAGES:
    print("  ", p.name)

6 image(s) found
   ad.png
   ad_1.png
   ad_12.png
   RYZE_AD.jpeg
   WhatsApp Image 2026-08-17 at 11.31.08 PM.jpeg
   WhatsApp Image 2026-08-17 at 11.31.09 PM.jpeg


## Ground truth

Required strings — what a human reads off each image. Only the
decision-relevant content, not every pixel: platform chrome, the music credit
and the comment box are noise, not ad copy.

Without this there is nothing to score either extractor against.

In [38]:
GROUND_TRUTH = {
    # Image 1 — "I Made $500K in 2 Minutes" reel
    "WhatsApp Image 2026-08-17 at 11.31.08 PM.jpeg": [
        "Day 7/30", "I Made", "$500K", "in 2 Minutes",
    ],

    # Image 2 — "WIN UP TO $900" reel
    "WhatsApp Image 2026-08-17 at 11.31.09 PM.jpeg": [
        "WIN UP TO $900", "BEGINNER FRIENDLY",
    ],

    # Image 3 — Cas.Fyn tweet (UGC ads / Kling)
    "ad.png": [
        "Cas.Fyn", "@FynCas",
        "MakeUGC", "Kling 3.0", "550 videos/day",
        "Fully realistic UGC ads", "cinematic lighting",
        "natural human motion", "powered by AI agents",
        "UGC cost: $1", "Production time: minutes", "Scale: instant",
        "One AI engine that creates, tests, and scales short-form ads",
        "Show more", "Product", "Image Prompt",
    ],

    # Image 4 — ALEX SUZUKI tweet ($20K/month)
    "ad_1.png": [
        "ALEX SUZUKI", "@X_FINALBOSS",
        "you are COOKED if you can't make $20K+ per month online in 2026",
        "use AI to generate 300 posts in 15 minutes",
        "millions of views without spending a penny on",
        "put words in a document and sell it 1000x for $50+ each",
        "you can hire workers for", "Show more",
    ],

    # Image 5 — Chime checking account
    "ad_12.png": [
        "Open a Chime", "Checking", "Account in 2 minutes",
        "we'll spot you up to", "$200",
        "chime", "DEBIT", "VISA",
        "SpotMe", "eligibility requirements and overdraft limits apply",
    ],

    # Image 6 — RYZE mushroom coffee
    "RYZE_AD.jpeg": [
        "POTBELLY", "TRY RYZE",
        "January", "February", "March",
        "SPECIAL DEAL", "$27", "$45", "THIS WEEK ONLY",
        "RYZE", "MUSHROOM COFFEE",
        "healthymushroomcoffee",
        "RYZE is powered by functional mushrooms and adaptogens",
    ],
}

missing = [p.name for p in IMAGES if p.name not in GROUND_TRUTH]
if missing:
    print(f"unlabelled (extracted but not scored): {missing}")
print(f"{len(GROUND_TRUTH)} image(s) labelled")

6 image(s) labelled


## Prompts

Two calls, deliberately separated.

`TRANSCRIBE_PROMPT` is the OCR replacement — literal text only. Its output goes
to the classifier exactly as OCR output would.

`CLAIM_PROMPT` extracts the brand and what the ad promises, phrased as a search
question, for the review layer. It states what the ad *claims*, never whether
the claim is true.

In [39]:
CLAIM_PROMPT = """You are looking at an advertisement that someone has just seen.
They are wondering whether to trust it, and they want to find out what other
people's real experience with this product has been.

Output exactly four lines:

BRAND: the brand or product name as written in the ad, or NONE
PROMISE: the outcome the ad says or shows the product will deliver, or NONE
QUERY: the search a cautious buyer would type to find out whether other people
actually got that outcome
NEEDS: if QUERY is NONE, what the user could supply that would make a lookup
possible -- the caption text, the account name, the link, or a recording of the
full video. Otherwise NONE.

Rules for QUERY:
- Name the brand and its product category explicitly.
- Aim it at other people's experience, not at the company's own description.
- Use the everyday words a buyer would type, not the ad's marketing wording.
- Keep it under about twelve words.
- Write QUERY: NONE if there is no named product, company, app, or course to
  look up. A dollar figure, a personal claim, or a promise of income is not
  something that can be searched. Do not invent a generic how-to search in
  place of a real one.

Report only what the ad states or depicts. Do not evaluate the ad, do not say
whether the promise is true or plausible, and do not name any persuasion tactic.

Output only those four lines."""

## Extractors

In [18]:
import sys
print(sys.executable)
!"{sys.executable}" -m pip install openai

c:\Users\shash\AppData\Local\Programs\Python\Python312\python.exe



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
from openai import OpenAI
_client = OpenAI()

def _b64(path):
    return base64.b64encode(Path(path).read_bytes()).decode()

def _media_type(path):
    ext = Path(path).suffix.lower()
    return {
        ".png": "image/png",
        ".webp": "image/webp",
        ".gif": "image/gif",
    }.get(ext, "image/jpeg")

def vlm_call(path, prompt, model=MODEL):
    t0 = time.time()
    resp = _client.chat.completions.create(
        model=model,
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image_url",
                 "image_url": {"url": f"data:{_media_type(path)};base64,{_b64(path)}"}},
                {"type": "text", "text": prompt},
            ],
        }],
    )
    return resp.choices[0].message.content.strip(), time.time() - t0

def run_vlm(path):
    text, secs = vlm_call(path, TRANSCRIBE_PROMPT)
    flat = " ".join(text.split())
    return {"text": flat, "lines": text.splitlines(), "seconds": secs}

def run_rapidocr(path):
    from rapidocr_onnxruntime import RapidOCR
    engine = RapidOCR()
    t0 = time.time()
    result, _ = engine(str(path))
    secs = time.time() - t0
    if not result:
        return {"text": "", "lines": [], "seconds": secs}
    lines = [r[1] for r in result]
    return {"text": " ".join(lines), "lines": lines, "seconds": secs}

def run_app_ocr(path):
    """The project's own OCR path, including its space-repair step.

    Calling rapidocr raw understates the current system: app/ocr.py repairs
    merged words before the text reaches the classifier. This is the honest
    baseline for the VLM to beat.
    """
    import sys
    sys.path.insert(0, str(Path(r"C:\ETH\deeplabv3plus\DeepLabV3Plus-Pytorch\Group-6-Final-Project") / "app"))
    import ocr as app_ocr
    t0 = time.time()
    out = app_ocr.extract_text(Path(path).read_bytes())
    return {"text": out.get("text", ""), "lines": out.get("lines", []),
            "seconds": time.time() - t0}

_paddle = None
def run_paddleocr(path):
    """PaddleOCR. API changed between 2.x and 3.x, so both are handled."""
    global _paddle
    from paddleocr import PaddleOCR
    if _paddle is None:
        try:
            _paddle = PaddleOCR(lang="en", show_log=False)      # 2.x
        except TypeError:
            _paddle = PaddleOCR(lang="en")                      # 3.x
    t0 = time.time()
    try:
        raw = _paddle.predict(str(path))                        # 3.x
    except AttributeError:
        raw = _paddle.ocr(str(path))                            # 2.x
    secs = time.time() - t0

    lines = []
    for page in (raw or []):
        if isinstance(page, dict):                              # 3.x returns dicts
            lines += list(page.get("rec_texts", []))
        else:
            for entry in (page or []):
                try:
                    lines.append(entry[1][0])
                except Exception:
                    continue
    return {"text": " ".join(lines), "lines": lines, "seconds": secs}


EXTRACTORS = {
    "RapidOCR (raw)": run_rapidocr,
    "app/ocr.py":     run_app_ocr,
    "PaddleOCR":      run_paddleocr,
    "VLM":            run_vlm,
}
print("extractors ready:", list(EXTRACTORS))

extractors ready: ['RapidOCR (raw)', 'app/ocr.py', 'PaddleOCR', 'VLM']


## Scoring

**Exact** — the required string is present as written.
**Spacing-damaged** — present only once spaces are stripped (`TRY RY ZE`), so a
word-boundary regex in the trigger lexicon will not match it.
**Missing** — absent entirely.

In [41]:
def norm(s):    return re.sub(r"\s+", " ", s.lower()).strip()
def despace(s): return re.sub(r"[^a-z0-9$]", "", s.lower())

def score(extracted, required):
    exact, spacing, missed = [], [], []
    nt, dt = norm(extracted), despace(extracted)
    for want in required:
        if norm(want) in nt:
            exact.append(want)
        elif despace(want) in dt:
            spacing.append(want)
        else:
            missed.append(want)
    return exact, spacing, missed

print("scorer ready")

scorer ready


## Run

In [42]:
results = {}

for img in IMAGES:
    print(f"\n{'='*72}\n{img.name}\n{'='*72}")
    required = GROUND_TRUTH.get(img.name)
    for name, fn in EXTRACTORS.items():
        try:
            out = fn(img)
        except Exception as e:
            print(f"{name:<18} FAILED: {e}")
            continue
        results.setdefault(img.name, {})[name] = out
        print(f"\n{name}  ({out['seconds']:.1f}s)")
        print(f"  {out['text'][:180]}")
        if required:
            e, s, m = score(out["text"], required)
            out["exact"], out["spacing"], out["missed"] = len(e), s, m
            print(f"  recovered {len(e)}/{len(required)}", end="")
            if s: print(f"  |  spacing-damaged: {s}", end="")
            if m: print(f"  |  MISSING: {m}", end="")
            print()


ad.png

RapidOCR (raw)  (2.8s)
  Cas.Fyn@FynCas·Jun11 MakeUGC&Kling 3.0-550videos/day Fully realistic UGC ads—cinematic lighting.natural human motion, clean pacing— powered by Al agents. -UGC cost:$1 -Production t
  recovered 10/16  |  spacing-damaged: ['550 videos/day', 'UGC cost: $1', 'Production time: minutes', 'Scale: instant']  |  MISSING: ['powered by AI agents', 'One AI engine that creates, tests, and scales short-form ads']

app/ocr.py  (1.2s)
  Cas.Fyn口 @FynCas-Jun11 MakeUGC&Kling3.0-550 videos/day Fully realistic UGC ads —cinematic lighting.natural human motion, clean pacing— powered by Al agents. UGC cost:S1 -Production
  recovered 12/16  |  spacing-damaged: ['Production time: minutes']  |  MISSING: ['powered by AI agents', 'UGC cost: $1', 'One AI engine that creates, tests, and scales short-form ads']
PaddleOCR          FAILED: Unknown argument: show_log

VLM  (2.2s)
  Cas.Fyn @FynCas · Jun 11 MakeUGC & Kling 3.0 = 550 videos/day Fully realistic UGC ads — cinematic lightin

## Comparison table — for the writeup

In [43]:
rows = []
for fname, by_engine in results.items():
    required = GROUND_TRUTH.get(fname)
    if not required: continue
    for name, out in by_engine.items():
        rows.append({
            "image": fname, "engine": name, "required": len(required),
            "exact": out.get("exact", 0),
            "spacing": len(out.get("spacing", [])),
            "missed": len(out.get("missed", [])),
            "seconds": round(out["seconds"], 1),
        })

if rows:
    print(f"{'engine':<18}{'req':>5}{'exact':>7}{'spacing':>9}{'missed':>8}{'sec':>7}")
    print("-" * 54)
    for name in EXTRACTORS:
        sub = [r for r in rows if r["engine"] == name]
        if not sub: continue
        req = sum(r["required"] for r in sub)
        ex  = sum(r["exact"] for r in sub)
        sp  = sum(r["spacing"] for r in sub)
        ms  = sum(r["missed"] for r in sub)
        sec = sum(r["seconds"] for r in sub) / len(sub)
        print(f"{name:<18}{req:>5}{ex:>7}{sp:>9}{ms:>8}{sec:>7.1f}")
        print(f"{'':<18}{'':<5}{ex/req:>6.0%}")

    print("\nmarkdown for the report:\n")
    print("| Extractor | Required strings | Recovered | Spacing-damaged | Missing | Mean seconds |")
    print("|---|---|---|---|---|---|")
    for name in EXTRACTORS:
        sub = [r for r in rows if r["engine"] == name]
        if not sub: continue
        req = sum(r["required"] for r in sub); ex = sum(r["exact"] for r in sub)
        print(f"| {name} | {req} | {ex} ({ex/req:.0%}) | "
              f"{sum(r['spacing'] for r in sub)} | {sum(r['missed'] for r in sub)} | "
              f"{sum(r['seconds'] for r in sub)/len(sub):.1f} |")

engine              req  exact  spacing  missed    sec
------------------------------------------------------
RapidOCR (raw)       53     29       19       5    2.0
                          55%
app/ocr.py           53     34       13       6    0.8
                          64%
VLM                  53     47        2       4    1.4
                          89%

markdown for the report:

| Extractor | Required strings | Recovered | Spacing-damaged | Missing | Mean seconds |
|---|---|---|---|---|---|
| RapidOCR (raw) | 53 | 29 (55%) | 19 | 5 | 2.0 |
| app/ocr.py | 53 | 34 (64%) | 13 | 6 | 0.8 |
| VLM | 53 | 47 (89%) | 2 | 4 | 1.4 |


## Does it change the verdict?

Extraction accuracy only matters if it changes what the user is told. This runs
each extractor's output through the classifier and the trigger lexicon.

In [44]:
import sys
REPO = Path(r"C:\ETH\deeplabv3plus\DeepLabV3Plus-Pytorch\Group-6-Final-Project")
sys.path.insert(0, str(REPO / "app"))
import predict, tactics

for fname, by_engine in results.items():
    print(f"\n{fname}")
    for name, out in by_engine.items():
        if not out["text"].strip():
            continue
        p = predict.predict(out["text"])
        rows_ = tactics.build_tactics(p, out["text"])
        findings = [r for r in rows_ if not r["uncertain"]]
        phrases = [ph["text"] for r in rows_ for ph in r["phrases"]]
        print(f"  {name:<18} {p['label']:<22}{p['confidence']:>6.0%}   "
              f"{len(findings)} finding(s)  phrases: {phrases or 'none'}")


ad.png
  RapidOCR (raw)     Exaggerated Claims       53%   0 finding(s)  phrases: none
  app/ocr.py         Exaggerated Claims       51%   0 finding(s)  phrases: none
  VLM                Social Proof             56%   0 finding(s)  phrases: none

ad_1.png
  RapidOCR (raw)     Social Proof             53%   0 finding(s)  phrases: none
  app/ocr.py         Social Proof             57%   0 finding(s)  phrases: none
  VLM                Social Proof             86%   1 finding(s)  phrases: none

ad_12.png
  RapidOCR (raw)     Social Proof             40%   0 finding(s)  phrases: none
  app/ocr.py         Social Proof             38%   0 finding(s)  phrases: none
  VLM                Urgency                  34%   0 finding(s)  phrases: none

RYZE_AD.jpeg
  RapidOCR (raw)     FOMO                     39%   0 finding(s)  phrases: none
  app/ocr.py         FOMO                     35%   0 finding(s)  phrases: none
  VLM                FOMO                     37%   0 finding(s)  phrases: no

## Claim extraction for the review layer

Second call. Output feeds query generation — the brand plus what the ad
promises. Note the prompt asks what the ad *claims*, never whether it is true.

In [45]:
for img in IMAGES:
    try:
        out, secs = vlm_call(img, CLAIM_PROMPT)
    except Exception as e:
        print(f"{img.name}: FAILED {e}")
        continue

    brand = promise = query = None
    for line in out.splitlines():
        up = line.upper()
        if up.startswith("BRAND:"):   brand   = line.split(":", 1)[1].strip()
        elif up.startswith("PROMISE:"): promise = line.split(":", 1)[1].strip()
        elif up.startswith("QUERY:"):   query   = line.split(":", 1)[1].strip()

    print(f"{img.name}  ({secs:.1f}s)")
    print(f"  brand:   {brand}")
    print(f"  promise: {promise}")
    print(f"  query:   {query}")
    if query and query.upper() != "NONE":
        print(f"  -> search: {query}")
    print()

ad.png  (1.2s)
  brand:   Kling 3.0
  promise: Creates, tests, and scales short-form ads automatically.
  query:   Kling 3.0 ad creation service reviews
  -> search: Kling 3.0 ad creation service reviews

ad_1.png  (0.9s)
  brand:   NONE
  promise: Make $20K+ per month online in 2026
  query:   NONE

ad_12.png  (2.5s)
  brand:   Chime
  promise: Spot you up to $200 for opening a checking account
  query:   Do Chime checking accounts really spot you $200?
  -> search: Do Chime checking accounts really spot you $200?

RYZE_AD.jpeg  (1.3s)
  brand:   RYZE
  promise: Potbelly reduction over three months
  query:   RYZE mushroom coffee reviews potbelly results
  -> search: RYZE mushroom coffee reviews potbelly results

WhatsApp Image 2026-08-17 at 11.31.08 PM.jpeg  (1.1s)
  brand:   NONE
  promise: $500K in 2 minutes
  query:   NONE

WhatsApp Image 2026-08-17 at 11.31.09 PM.jpeg  (1.3s)
  brand:   NONE
  promise: Win up to $900, beginner friendly
  query:   NONE



## Notes for the writeup

- The VLM replaces OCR only. Classification remains the fine-tuned DistilBERT;
  the prompt forbids interpretation, tactic naming, and any judgement of the ad.
- Report the comparison table and, more importantly, the downstream effect: how
  many trigger phrases fired and whether the reported verdict changed. Recovering
  `THIS WEEK ONLY` matters because it makes the Urgency trigger match, not
  because the string is longer.
- Cost and latency are real trade-offs against local OCR — record both.
- The VLM is a network dependency in the request path. Keep `app/ocr.py`'s
  existing backend as fallback so the demo survives a failed call.